In [2]:
!pip install medmnist timm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.6 MB/s eta 0:00:00


In [37]:
import torch
import timm
import medmnist
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
from medmnist import INFO
from torch.cuda.amp import autocast, GradScaler

print("==== Starting Script ====")

torch.backends.cudnn.benchmark = True

SyntaxError: unterminated string literal (detected at line 32) (data.py, line 32)

In [50]:
# ==========================
# DEVICE DEBUG
# ==========================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", torch.cuda.get_device_properties(0).total_memory/1e9, "GB")

Device: cuda
GPU: Tesla T4
GPU Memory: 15.637086208 GB


In [53]:
# ==========================
# DATASET INFO
# ==========================

print("\nLoading dataset info...")

data_flag = "pathmnist"
info = INFO[data_flag]

print("Dataset:", data_flag)

DataClass = getattr(medmnist, info["python_class"])
num_classes = len(info["label"])

print("Number of classes:", num_classes)


Loading dataset info...
Dataset: pathmnist
Number of classes: 9


In [ ]:
# ==========================
# DATALOADER
# ==========================

print("\nCreating dataloaders...")

batch_size = 32
num_workers = 2

train_loader, val_loader, test_loader = get_dataloaders(
    train_dataset, val_dataset, test_dataset, batch_size=batch_size, num_workers=num_workers
)

print("Dataloaders ready")

In [42]:
# ==========================
# SANITY CHECK BATCH
# ==========================

print("\nChecking first batch...")

images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)


Checking first batch...
Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32, 1])


In [43]:
# ==========================
# MODEL
# ==========================

print("\nLoading model...")

model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=num_classes
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())

print("Model loaded")
print("Total parameters:", total_params)


Loading model...


Model loaded
Total parameters: 4019077


In [44]:
# ==========================
# LOSS + OPTIMIZER
# ==========================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

scaler = GradScaler()

print("Optimizer + scheduler ready")

Optimizer + scheduler ready


/content/medmnist_efficientnet_project/src/model.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [47]:
# ==========================
# TRAINING LOOP
# ==========================

print("\n==== TRAINING START ====")

epochs = 5
best_acc = 0

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss = train_epoch()
    val_acc = validate()

    scheduler.step()

    print("Train Loss:", round(train_loss,4))
    print("Val Acc:", round(val_acc,4))

    if val_acc > best_acc:

        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("Model saved")

print("\nTraining finished")
print("Best accuracy:", best_acc)


==== TRAINING START ====

Epoch 1/5


TypeError: train_epoch() takes 0 positional arguments but 6 were given

In [48]:
test_dataset = DataClass(split="test", transform=val_tfms, download=True)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

def test_model():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.squeeze().long().to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print("Test Accuracy:", correct / total)

model.load_state_dict(torch.load("best_model.pth"))
test_model()

TypeError: test_model() takes 0 positional arguments but 3 were given

In [49]:
import matplotlib.pyplot as plt
import torch

model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

def show_prediction(dataset, idx=0):
    img, label = dataset[idx]

    if hasattr(label, "__len__"):
        label = int(label[0])
    else:
        label = int(label)

    x = img.unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(x)
        pred = torch.argmax(out, dim=1).item()

    img_np = img.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * 0.5) + 0.5
    img_np = img_np.clip(0, 1)

    plt.figure(figsize=(4, 4))
    plt.imshow(img_np)
    plt.title(f"True: {info['label'][str(label)]}\nPred: {info['label'][str(pred)]}")
    plt.axis("off")
    plt.show()

show_prediction(test_dataset, idx=10)

TypeError: show_prediction() got multiple values for argument 'idx'